# Library 04: building the networks

Here the partnership table becomes **one network per format**. We never mix formats in one network.

* **Node:** an England batter with at least one partnership in that format.
* **Edge:** two batters who shared at least one partnership. The network is undirected and one-mode
  (monomodal): every node is a batter.
* **Edge weight:** the total partnership runs of the pair, over all their partnerships in that format. This
  makes the networks valued (weighted) networks.

The networks are built only from the partnership table, so we can rebuild them from it at any time.

Needs `lib_00_settings`.

In [ ]:
PAIR_KEY_COLUMNS = ["format", "player_1_id", "player_2_id"]


def build_player_table(observations: pd.DataFrame) -> pd.DataFrame:
    """One row per player ID with the name we show and every spelling of the name we came across.

    Edge case: the same player can be spelled differently in different files (e.g. with or without a dot).
    Because we group by ID, these spellings are merged, and we show the spelling used most often.
    """
    both_positions = pd.concat([
        observations[["player_1_id", "player_1_name"]].set_axis(["player_id", "name"], axis=1),
        observations[["player_2_id", "player_2_name"]].set_axis(["player_id", "name"], axis=1),
    ])
    name_counts = (both_positions.value_counts().reset_index(name="times_seen")
                   .sort_values(["player_id", "times_seen", "name"], ascending=[True, False, True]))
    display_name = name_counts.drop_duplicates("player_id").set_index("player_id")["name"]
    all_spellings = name_counts.groupby("player_id")["name"].agg(lambda names: "|".join(sorted(set(names))))
    return pd.DataFrame({"player_name": display_name, "name_variants": all_spellings}).reset_index()


def build_edge_table(observations: pd.DataFrame, player_table: pd.DataFrame) -> pd.DataFrame:
    """One row per format and pair of batters (player_1_id is always the smaller ID)."""
    observations_with_key = observations.copy()
    # An innings is only unique together with its match, so we build a combined key for counting innings.
    observations_with_key["innings_key"] = (observations_with_key["match_id"].astype(str) + "#"
                                            + observations_with_key["innings_index"].astype(str))
    pair_groups = observations_with_key.groupby(PAIR_KEY_COLUMNS, sort=True)
    edge_table = pair_groups.agg(
        weight=("partnership_runs", "sum"),                 # the edge weight: total partnership runs
        partnership_count=("partnership_runs", "size"),     # how many separate partnerships
        total_matches=("match_id", "nunique"),
        total_innings=("innings_key", "nunique"),
        first_date=("match_date", "min"),
        last_date=("match_date", "max"),
        mean_partnership_runs=("partnership_runs", "mean"),
        median_partnership_runs=("partnership_runs", "median"),
        min_partnership_runs=("partnership_runs", "min"),
        max_partnership_runs=("partnership_runs", "max"),
    ).reset_index()
    name_of_player = player_table.set_index("player_id")["player_name"]
    edge_table.insert(2, "player_1_name", edge_table["player_1_id"].map(name_of_player))
    edge_table.insert(4, "player_2_name", edge_table["player_2_id"].map(name_of_player))
    edge_table["pair_key"] = edge_table["player_1_id"] + "__" + edge_table["player_2_id"]
    return edge_table


def build_node_table(observations: pd.DataFrame, player_table: pd.DataFrame) -> pd.DataFrame:
    """One row per format and batter who was part of at least one partnership.

    What the columns mean:
    * matches_batted / innings_batted: number of matches / innings in which the batter had a partnership
    * partnership_observations: number of partnerships the batter was part of
    * distinct_partners: number of different partners (this is the same as the node's degree)
    * first_date / last_date: first and last match with a partnership in this format
    * median_entry_partnership: in a typical innings, the number of the first partnership the batter joined
      (1 = opened the batting, 3 = came in at the fall of the second wicket, ...). We only use it to describe
      where a batter usually came in.
    """
    key_columns = ["format", "match_id", "innings_index", "match_date", "partnership_seq"]
    # We look at each partnership twice, once from each player's side, so every player gets their own rows.
    as_player_1 = observations[key_columns + ["player_1_id", "player_2_id"]].set_axis(key_columns + ["player_id", "partner_id"], axis=1)
    as_player_2 = observations[key_columns + ["player_2_id", "player_1_id"]].set_axis(key_columns + ["player_id", "partner_id"], axis=1)
    player_view = pd.concat([as_player_1, as_player_2], ignore_index=True)
    player_view["innings_key"] = player_view["match_id"].astype(str) + "#" + player_view["innings_index"].astype(str)

    node_table = player_view.groupby(["format", "player_id"], sort=True).agg(
        matches_batted=("match_id", "nunique"),
        innings_batted=("innings_key", "nunique"),
        partnership_observations=("partner_id", "size"),
        distinct_partners=("partner_id", "nunique"),
        first_date=("match_date", "min"),
        last_date=("match_date", "max"),
    ).reset_index()
    entry_point = (player_view.groupby(["format", "player_id", "innings_key"])["partnership_seq"].min()
                   .groupby(["format", "player_id"]).median().rename("median_entry_partnership").reset_index())
    node_table = node_table.merge(entry_point, on=["format", "player_id"], how="left", validate="one_to_one")
    node_table.insert(2, "player_name", node_table["player_id"].map(player_table.set_index("player_id")["player_name"]))
    return node_table

## From tables to a graph

In [ ]:
def build_format_graph(node_table: pd.DataFrame, edge_table: pd.DataFrame, format_name: str) -> nx.Graph:
    """Undirected, weighted, one-mode graph for one format."""
    partnership_graph = nx.Graph(format=format_name)
    for node in node_table[node_table["format"] == format_name].itertuples(index=False):
        partnership_graph.add_node(
            node.player_id, player_id=node.player_id, player_name=node.player_name,
            format=format_name, matches_batted=int(node.matches_batted), innings_batted=int(node.innings_batted),
            partnership_observations=int(node.partnership_observations),
            distinct_partners=int(node.distinct_partners),
            first_date=str(node.first_date), last_date=str(node.last_date),
            median_entry_partnership=float(node.median_entry_partnership))
    for edge in edge_table[edge_table["format"] == format_name].itertuples(index=False):
        partnership_graph.add_edge(
            edge.player_1_id, edge.player_2_id,
            weight=int(edge.weight), partnership_count=int(edge.partnership_count),
            total_matches=int(edge.total_matches), total_innings=int(edge.total_innings))
    return partnership_graph


def load_format_graph(format_name: str) -> nx.Graph:
    """Rebuild the graph of one format from the node and edge tables saved in Stage 5."""
    file_tag = FORMAT_FILE_TAG[format_name]
    node_table = read_table(PROCESSED_FOLDER / f"nodes_{file_tag}.csv")
    edge_table = read_table(PROCESSED_FOLDER / f"edges_{file_tag}.csv")
    return build_format_graph(node_table, edge_table, format_name)

## Checks on the networks

These checks make sure the networks say exactly what the partnership table says: same players, same pairs,
same runs.

In [ ]:
def aggregation_checks(observations: pd.DataFrame, node_table: pd.DataFrame, edge_table: pd.DataFrame,
                       format_name: str) -> list[tuple[str, bool, str]]:
    """Return (name, passed, detail) for every check on one format's network."""
    format_observations = observations[observations["format"] == format_name]
    format_edges = edge_table[edge_table["format"] == format_name]
    format_nodes = node_table[node_table["format"] == format_name]
    partnership_graph = build_format_graph(node_table, edge_table, format_name)
    pair_groups = format_observations.groupby(["player_1_id", "player_2_id"])
    check_results: list[tuple[str, bool, str]] = []

    def record(check_name: str, passed: bool, detail: str = "") -> None:
        check_results.append((check_name, bool(passed), detail))

    record("no_self_loops", (format_edges["player_1_id"] != format_edges["player_2_id"]).all()
           and nx.number_of_selfloops(partnership_graph) == 0)
    record("canonical_pair_order", (format_edges["player_1_id"] < format_edges["player_2_id"]).all())
    record("edge_count_equals_unique_pairs",
           len(format_edges) == pair_groups.ngroups == partnership_graph.number_of_edges(),
           f"{len(format_edges)} edges / {pair_groups.ngroups} pairs")
    edges_by_pair = format_edges.set_index(["player_1_id", "player_2_id"])
    record("weight_equals_sum_of_observation_runs",
           edges_by_pair["weight"].equals(pair_groups["partnership_runs"].sum().reindex(edges_by_pair.index)))
    record("partnership_count_equals_observation_count",
           (edges_by_pair["partnership_count"] == pair_groups.size().reindex(edges_by_pair.index)).all())
    record("total_matches_correct",
           (edges_by_pair["total_matches"] == pair_groups["match_id"].nunique().reindex(edges_by_pair.index)).all())
    with_innings_key = format_observations.assign(
        innings_key=format_observations["match_id"].astype(str) + "#" + format_observations["innings_index"].astype(str))
    innings_per_pair = with_innings_key.groupby(["player_1_id", "player_2_id"])["innings_key"].nunique()
    record("total_innings_correct", (edges_by_pair["total_innings"] == innings_per_pair.reindex(edges_by_pair.index)).all())
    record("sum_of_weights_equals_sum_of_observation_runs",
           int(format_edges["weight"].sum()) == int(format_observations["partnership_runs"].sum()),
           f"{int(format_edges['weight'].sum())} runs")
    record("every_edge_has_observations", (format_edges["partnership_count"] >= 1).all())
    players_seen = set(format_observations["player_1_id"]) | set(format_observations["player_2_id"])
    record("node_set_equals_players_in_observations",
           set(format_nodes["player_id"]) == players_seen == set(partnership_graph.nodes), f"{len(format_nodes)} nodes")
    degree_of = dict(partnership_graph.degree())
    record("distinct_partners_equals_degree",
           all(degree_of[node.player_id] == node.distinct_partners for node in format_nodes.itertuples()))
    # Every partnership has two players, so the per-player counts must add up to twice the number of partnerships.
    record("node_observations_sum_is_twice_observations",
           int(format_nodes["partnership_observations"].sum()) == 2 * len(format_observations))
    return check_results